# DTAT391B. deeptrack.sources.folder

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/3-advanced-topics/DTAT391B_sources.folder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

This advanced tutorial introduces the sources.folder module.

## 1. What is `folder.py`?

The `folder.py` module enables the management of image datasets organized in a directory hierarchy. It contains a single class `ImageFolder` that provides utilities to perform structured naming, organization, and retrieval of image data.

The key roles of `folder.py` are:

- **Recursively Index Folder-Based Datasets:**
  The module provides utilities to index datasets that are stored on disk in folders,
  typically structured by class labels or categories. It allows automatic discovery
  and registration of files into sources, preserving the folder hierarchy.

- **Associate Metadata with Files:**
  It supports extracting metadata from folder structures (e.g., parent folder as label)
  and saving them as fields in the resulting `Source`, making them directly usable
  in downstream pipelines.

- **Dynamic File Access for Streaming:**
  Instead of loading all file contents at once, `folder.py` allows for lazy evaluation
  by returning file paths as source fields. This enables streaming images or data
  directly into DeepTrack features (e.g., `dt.LoadImage`), supporting scalable workflows.

- **Split and Augment Datasets:**
  It works seamlessly with `random_split`, `Product`, and other tools in `base.py`,
  allowing flexible construction of training, validation, and test sets with dynamic
  or constant augmentation fields.

## 2. Creating a Directory Structure

Since the `ImageFolder` class expects images to be stored in directories categorized by class names, you will need to create a dummy directory structure for demonstration purposes.

In [2]:
import os
import shutil

# Define root directory
dataset_path = "dummy_dataset"

# Remove existing directory if exists
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)

# Define splits and class names
splits = ["train", "test"]
classes = ["cat", "dog", "bird"]

# Create directory structure and add dummy files
for split in splits:
    for class_name in classes:
        class_dir = os.path.join(dataset_path, split, class_name)
        os.makedirs(class_dir, exist_ok=True)
        for i in range(3): 
            with open(
                os.path.join(class_dir, f"{split}_image_{i}.jpg"), "w"
            ) as f:
                f.write("")

Print the directory structure.

In [3]:
for root, dirs, files in os.walk(dataset_path):

    # Get depth of directory for indenting the print text
    depth = root.replace(dataset_path, "").count(os.sep)
    indent = "  " * depth

    # Directories
    directory_name = os.path.basename(root)
    print(f"{indent}📂 {directory_name}")
    
    # Files
    for filename in sorted(files):
        print(f"{indent}  📄 {filename}")

📂 dummy_dataset
  📂 test
    📂 cat
      📄 test_image_0.jpg
      📄 test_image_1.jpg
      📄 test_image_2.jpg
    📂 dog
      📄 test_image_0.jpg
      📄 test_image_1.jpg
      📄 test_image_2.jpg
    📂 bird
      📄 test_image_0.jpg
      📄 test_image_1.jpg
      📄 test_image_2.jpg
  📂 train
    📂 cat
      📄 train_image_0.jpg
      📄 train_image_1.jpg
      📄 train_image_2.jpg
    📂 dog
      📄 train_image_0.jpg
      📄 train_image_1.jpg
      📄 train_image_2.jpg
    📂 bird
      📄 train_image_0.jpg
      📄 train_image_1.jpg
      📄 train_image_2.jpg


## 3.  Initializing an `ImageFolder`.
Now that the dummy directory is created, initialize an `ImageFolder` object.

In [4]:
from deeptrack.sources.folder import ImageFolder

train_source = ImageFolder(os.path.join(dataset_path, "train"))

Print total number of images:

In [5]:
len(train_source)

9

Print class names:

In [6]:
train_source.classes

['dog', 'bird', 'cat']

## 4. Getting Category Names from File Paths


Prepare an example path to one of the files:

In [7]:
example_path = os.path.join(dataset_path, "train", "dog", "image_1.jpg")
example_path

'dummy_dataset/train/dog/image_1.jpg'

Get the corresponding category name:

In [8]:
category = train_source.get_category_name(example_path, directory_level=0)
category

'dog'

## 5. Splitting the Dataset

If the dataset has subcategories (e.g., dog/cat/bird), you can split it according to those subcategories.

Create the top-level ImageFolder from the dataset root:

In [9]:
dataset = ImageFolder("dummy_dataset")

Split into training and test subsets using top-level folder names:

In [10]:
train_source, test_source = dataset.split("train", "test")

Print number of samples in each subset:

In [11]:
print(f"Train samples: {len(train_source)}")

Train samples: 9


In [12]:
print(f"Test samples: {len(test_source)}")

Test samples: 9


Print available classes:

In [13]:
print("Train classes:", train_source.classes)

Train classes: ['dog', 'bird', 'cat']


In [14]:
print("Test classes:", test_source.classes)

Test classes: ['dog', 'bird', 'cat']


List all image paths in train:

In [15]:
for item in train_source:
    print(item["path"])

dummy_dataset/train/bird/train_image_0.jpg
dummy_dataset/train/bird/train_image_1.jpg
dummy_dataset/train/bird/train_image_2.jpg
dummy_dataset/train/cat/train_image_0.jpg
dummy_dataset/train/cat/train_image_1.jpg
dummy_dataset/train/cat/train_image_2.jpg
dummy_dataset/train/dog/train_image_0.jpg
dummy_dataset/train/dog/train_image_1.jpg
dummy_dataset/train/dog/train_image_2.jpg


List all image paths in test:

In [16]:
for item in test_source:
    print(item["path"])

dummy_dataset/test/bird/test_image_0.jpg
dummy_dataset/test/bird/test_image_1.jpg
dummy_dataset/test/bird/test_image_2.jpg
dummy_dataset/test/cat/test_image_0.jpg
dummy_dataset/test/cat/test_image_1.jpg
dummy_dataset/test/cat/test_image_2.jpg
dummy_dataset/test/dog/test_image_0.jpg
dummy_dataset/test/dog/test_image_1.jpg
dummy_dataset/test/dog/test_image_2.jpg
